# Exercise 06: Systems of linear equations 2

In [ ]:
import numpy as np
np.set_printoptions(precision=5)
import matplotlib.pyplot as plt

## Pivoting

#### Let us recall the Gaussian elimination algorithm from last time:

In [ ]:
def gauss_elimination(A, b):
    """Gaussian elimination of the augmented matrix [A|b]."""
    Ab = np.column_stack((A, b))

    for p, pivot_row in enumerate(Ab[:-1]):
        for row in Ab[p+1:]:
            if row[p]:
                m = row[p]/pivot_row[p]
                row[p:] = row[p:] - pivot_row[p:] * m
    return Ab

In [ ]:
def gauss_solution(Ab):
    """Solution of the system of equations after Gaussian elimination - back substitution."""
    n = Ab.shape[0]
    x = np.zeros(n)
    A = Ab[:, :-1].copy()
    b = Ab[:, -1].copy()
    for i in range(n)[::-1]:
        x[i] = (b[i] - np.dot(A[i, i+1:], x[i+1:])) / A[i, i]
    return x

### Gaussian elimination with partial pivoting

The procedure is the same as the Gaussian elimination method we already know, with the following addition:
* After each selection of the pivot row ``i`` in the first ``for`` loop, among all rows $j \geq p$ we find the index ``i_max`` of the row that has the element with the largest absolute value in column ``i``.
* If the initially selected row ``i`` is not the row with the largest value in column ``i``, we swap the two rows using advanced slicing:

    ``Ab[[i], :], Ab[[i_max], :] = Ab[[i_max], :], Ab[[i], :]``


* We continue with Gaussian elimination, where the new row ``i`` is our pivot row.

---

**Question 1:** Prepare a function ``gauss_elimination_pivoting`` by extending the existing function ``gauss_elimination`` with partial pivoting according to the procedure described above.

**Question 2:** The force and moment equilibrium equations for the beam in the figure are:

<img src="konzola.png" style="width:550px">

$$
\begin{align}
\text{direction }x &:\quad F\,\cos(\alpha) + A_x = 0\\
\text{direction }y &:\quad F\,\sin(\alpha) = A_y\\
\text{moment about }A&:\quad M + F\,\sin(\alpha)\,l = M_A
\end{align}
$$

We know the force $A_y$ and the moment $M_A$ at the support, and the length of the beam is also known. Using an appropriate method, determine the moment $M$, the force $F$ and the reaction $A_x$, for the vector of unknowns $[M, F,  A_x]$.

In [ ]:
# Data, Question 2
L = 1 # m
M_A = 9.5 # Nm
A_y = 12 # N
alpha = np.pi/4 # rad

---

## LU decomposition

---

**Example:** We compute the support reactions of the beam for different angles $\varphi$:

<img src="nosilec.png" style="width:550px">

Equilibrium equations:
$$
\begin{align}
\text{direction }x &:\quad  A_x = F\,\cos(\varphi)\\
\text{direction }y &:\quad A_y + B = F\,\sin(\varphi)\\
\text{moment about }A&:\quad Bl = Fa\,\sin(\varphi)
\end{align}
$$

For different values of the angle $\varphi$, which appears only on the right-hand sides of the equations, $\mathbf{b}$ changes, while the matrix $\mathbf{A}$ stays the same all the time.

---

### Procedure for solving the resulting system

### Doolittle decomposition

### LU decomposition, Gaussian elimination in ``scipy``

---

**Question 3:** Using the beam from Question 2, demonstrate solving a system of linear equations with LU decomposition, using functions from the ``scipy`` package. How would you obtain the result of Gaussian elimination with partial pivoting using one of the functions from the ``scipy.linalg`` package?

**Question 4:** The force and moment equilibrium equations for the beam in the figure are:

<img src="nosilec.png" style="width:550px">

Determine the support reactions using the functions ``lu_factor`` and ``lu_solve`` from `scipy.linalg`, for five different angles $\varphi$ from the given list, performing the decomposition of the coefficient matrix only once. Print the solution each time with the ``print`` function.

$$
\begin{align}
\text{direction }x &:\quad  A_x = F\,\cos(\varphi)\\
\text{direction }y &:\quad A_y + B = F\,\sin(\varphi)\\
\text{moment about }A&:\quad Bl = Fa\,\sin(\varphi)
\end{align}
$$

In [ ]:
# Data, Question 4
L = 1.5 # m
a = 0.55 # m
F = 85 # N
phi_list = [0, np.pi/6, np.pi/4, np.pi/3, np.pi/2]

---

## Computing the inverse of a matrix

Computing the inverse of the matrix $\mathbf{A}$ essentially means solving the system of equations:
$$\mathbf{AX} = \mathbf{I}$$

We are therefore looking for a matrix $\mathbf{X}$ that gives the identity matrix when multiplied by $\mathbf{A}$. Hence $\mathbf{X} = \mathbf{A}^{-1}$. This amounts to solving several systems of linear equations of the form:

$$\mathbf{A}\mathbf{x}_i = b_i$$

where $x_i$ and $b_i$ are the $i$-th columns of the matrices $\mathbf{X}$ and $\mathbf{I}$ (from the definition of matrix multiplication).

---

**Question 5:** Use any existing function for solving a system of linear equations and, following the procedure described above, determine the inverse of the matrix $\mathbf{A}$ from Question 4.

---

### Solving overdetermined systems - pseudo-inverse

---

**Question 6:** Determine the approximating line that describes the $n$ given points $(X, Y)$ with the least squared error. Find the parameters $k$ and $n$ of the approximating line $y = kx+n$ by solving the system of linear equations:

$$\mathbf{[X, 1]}_{~n\times 2}~\mathbf{x} = \mathbf{[Y]}$$

where the first column of the coefficient matrix contains the $X$ coordinates of the given points and the second column contains only ones. The right-hand side vector is equal to the vector of $Y$ coordinates of the points. Plot the set of points $(X, Y)$ and the obtained approximating line in the same figure.

In [ ]:
# Data, Question 6
X = np.linspace(1, 10, 100)
Y = 0.7*X - 15.5 + np.random.randn(len(X))
plt.plot(X, Y, 'b.')

We therefore have the system of equations:

$$k ~ x_1 + n = y_1$$
$$k ~ x_2 + n = y_2$$
$$\vdots$$
$$k ~ x_n + n = y_n$$

---

## Iterative methods for solving systems of linear equations

### Gauss-Seidel

In [ ]:
def gauss_seidel(A, b, x0, niter=10):
    """Gauss-Seidel iterative method."""
    deviations = []
    x = x0.copy()
    AA = A.copy()
    np.fill_diagonal(AA, 0)
    for i in range(niter):
        x_star = x.copy()
        for i in range(len(b)):
            x[i] = (b[i] - np.dot(AA[i], x)) / A[i,i]
        deviations.append(np.linalg.norm(x_star - x))
    return x, np.array(deviations)

---

**Question 7:** Solve the given system of equations using the iterative Gauss-Seidel method. Observe the effect of the number of iterations on the accuracy of the solution.

In [ ]:
# Data, Question 7
A = np.array([[-1, 2, 0, 0, 0, 0],
             [-1, 3, 1, 0, 0, 0],
             [0, -1, 4, 1, 0, 0],
             [0, 0, -2, 5, 0, 0],
             [0, 0, 0, 0, 5, 3],
             [0, 0, 0, 0, 1, -3]], dtype=float)

b = np.array([0, 18, 0, 12, 3, 5], dtype=float)